# Advanced WLASL ASL Video Classification - TensorFlow/Keras Google Colab Implementation

This notebook implements an efficient ASL word recognition system using MediaPipe landmarks on the WLASL dataset, optimized for Google Colab free tier. Follow each section for end-to-end training, evaluation, and deployment using landmark-based processing.

In [ ]:
# Cell 1: Setup & Environment Configuration
import tensorflow as tf
import os
import logging
import psutil
import GPUtil

# TensorFlow version and GPU detection
print("TensorFlow version:", tf.__version__)
print("GPUs Available:", tf.config.list_physical_devices('GPU'))

# Install required packages
!pip install opencv-python-headless kaggle wandb mediapipe pyarrow pandas scipy joblib

# Memory optimization & growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Suppress warnings and set logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# System info display
print("RAM Available:", round(psutil.virtual_memory().available/1e9,2), "GB")
print("Storage Info:", round(psutil.disk_usage('/mnt').free/1e9,2), "GB free")

In [ ]:
# Cell 2: Data Download & Initial Setup
from google.colab import drive
drive.mount('/content/drive')

# Kaggle API setup
import json
with open('/content/drive/MyDrive/kaggle.json','r') as f:
    creds = json.load(f)
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']

# Download dataset
!kaggle datasets download -d risangbaskoro/wlasl-processed -p /content/data --unzip -q

# Directory structure and initial stats
!ls /content/data
import glob
video_paths = glob.glob('/content/data/videos/**/*.mp4', recursive=True)
print(f"Total videos found: {len(video_paths)}")

# Load metadata JSON
with open('/content/data/WLASL_v0.3.json','r') as f:
    metadata = json.load(f)
print("Total entries in metadata:", len(metadata['data']))

In [ ]:
# Cell 3: Data Exploration & Analysis
import pandas as pd
import matplotlib.pyplot as plt

# Normalize metadata JSON
df_meta = pd.json_normalize(metadata['data'])
display(df_meta.head())

# Class distribution top-20
class_counts = df_meta['gloss'].value_counts().head(20)
plt.figure(figsize=(10,4))
class_counts.plot.bar()
plt.title('Top 20 ASL Word Classes')
plt.show()

# Video duration and frame counts stub (using OpenCV)
import cv2
durations = []
for p in video_paths[:100]:
    cap = cv2.VideoCapture(p)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    durations.append(frames/fps if fps else 0)
cap.release()
pd.Series(durations).describe()

In [ ]:
# Cell 4: MediaPipe Landmark Extraction Pipeline
import mediapipe as mp
import numpy as np
from tqdm import tqdm

mp_holistic = mp.solutions.holistic.Holistic(static_image_mode=False)
def extract_landmarks(video_path, num_frames=32):
    import cv2
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total-1, num_frames).astype(int)
    landmarks = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: break
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = mp_holistic.process(image)
        frame_landmarks = []
        for lm_list in [results.left_hand_landmarks, results.right_hand_landmarks, results.face_landmarks, results.pose_landmarks]:
            if lm_list:
                frame_landmarks.extend([[lm.x, lm.y, lm.z, lm.visibility, lm.presence] for lm in lm_list.landmark])
            else:
                frame_landmarks.extend([[0]*5]* (len(lm_list.landmark) if lm_list else 0))
        landmarks.append(frame_landmarks)
    cap.release()
    return np.array(landmarks)

# Test extraction on first video
sample = extract_landmarks(video_paths[0])
print("Extracted landmark shape:", sample.shape)

In [ ]:
# Cell 5: Efficient Landmark Storage & Caching
import pyarrow as pa, pyarrow.parquet as pq

# Prepare Parquet table schema and write for first sample
video_id = os.path.basename(video_paths[0]).split('.')[0]
rows = []
for frame_idx, frame in enumerate(sample):
    for lm_idx, lm in enumerate(frame):
        rows.append({
            'video_id': video_id,
            'frame_index': int(frame_idx),
            'landmark_index': int(lm_idx),
            'x': float(lm[0]), 'y': float(lm[1]), 'z': float(lm[2]),
            'visibility': float(lm[3]), 'presence': float(lm[4])
        })
table = pa.Table.from_pylist(rows)
pq.write_table(table, 'landmarks_sample.parquet')

# Save NPY memmap for sample
np.save('landmarks_sample.npy', sample)
print("Parquet and NPY files created")

In [ ]:
# Cell 6: Landmark-Based Data Augmentation
import tensorflow as tf

def augment_landmarks(landmarks, noise_scale=0.01):
    # Add Gaussian noise and random drop
    noise = tf.random.normal(tf.shape(landmarks), stddev=noise_scale)
    dropped = tf.where(tf.random.uniform(tf.shape(landmarks))<0.05, 0.0, landmarks)
    return dropped + noise

# Test augmentation
aug_sample = augment_landmarks(tf.constant(sample, dtype=tf.float32))
print("Augmented landmark shape:", aug_sample.shape)

In [ ]:
# Cell 7: Model Architecture Implementation
from tensorflow.keras import layers, models

def build_landmark_model(sequence_length=32, n_landmarks=543, n_features=5, num_classes=2000):
    inputs = layers.Input(shape=(sequence_length, n_landmarks*n_features))
    x = layers.LayerNormalization()(inputs)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Attention()([x, x])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

model = build_landmark_model(sequence_length=sample.shape[0],
                             n_landmarks=sample.shape[1],
                             n_features=sample.shape[2])
model.summary()

In [ ]:
# Cell 8: Training Configuration & Optimization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.mixed_precision import experimental as mixed_precision

# Mixed precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_policy(policy)

# Learning rate schedule
lr_schedule = CosineDecay(initial_learning_rate=1e-3, decay_steps=10000)
optimizer = Adam(learning_rate=lr_schedule)

model.compile(optimizer=optimizer,
              loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5)])


In [ ]:
# Cell 9: Callbacks & Monitoring Setup
import wandb
from wandb.keras import WandbCallback

wandb.init(project='wlasl-landmarks', config={'epochs':50, 'batch_size':32})
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_landmark_model.h5', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5),
    tf.keras.callbacks.TensorBoard(log_dir='./logs'),
    WandbCallback()
]

In [ ]:
# Cell 10: Landmark Data Loading & Preprocessing
import tensorflow as tf

def load_landmark_sequence(np_path):
    seq = np.load(np_path)
    return tf.reshape(seq, (seq.shape[0], -1))

# Create tf.data dataset stub
paths = tf.constant(['landmarks_sample.npy'])
labels = tf.constant([0])
dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
dataset = dataset.map(lambda p, l: (augment_landmarks(load_landmark_sequence(p)), l),
                      num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Cell 11: Training Execution
history = model.fit(dataset,
                    validation_data=dataset.take(1),
                    epochs=50,
                    callbacks=callbacks)

import matplotlib.pyplot as plt
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()

In [ ]:
# Cell 12: Model Evaluation & Deployment
import tensorflow_model_optimization as tfmot

# Pruning
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude
pruned_model = prune_low_magnitude(model, pruning_schedule=tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.0, final_sparsity=0.5, begin_step=2000, end_step=10000))

# TFLite conversion
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('model_landmarks.tflite', 'wb') as f:
    f.write(tflite_model)
print("TFLite model size (MB):", round(os.path.getsize('model_landmarks.tflite')/1e6,2))